In [ ]:
"""
🌾 Professional Crop Disease Diagnosis System
COMPLETE FIXED VERSION - Multi-Language + Accurate Image Recognition

Features:
- 11 Indian Languages with FULL UI translation
- Native language keyboards for each language
- Accurate crop disease detection
- Photo Upload (PNG/JPG/JPEG only)
- Camera Capture
- Zoom slider for results
- Professional Modern GUI
"""

import customtkinter as ctk
import cv2
from PIL import Image
import requests
import base64
import os
import logging
from tkinter import filedialog, messagebox
from pathlib import Path
from typing import Optional
from datetime import datetime
import tempfile
from threading import Lock, Thread
from dataclasses import dataclass, field

# ============================================================================
# CONFIGURATION
# ============================================================================

@dataclass
class AppConfig:
    """Application Configuration"""
    OLLAMA_HOST: str = os.getenv("OLLAMA_HOST", "http://localhost:11434")
    OLLAMA_TIMEOUT: tuple = (10, 180)
    OLLAMA_MODEL: str = os.getenv("OLLAMA_MODEL", "llava:7b")
    
    MAX_FILE_SIZE_MB: int = 50
    ALLOWED_EXTENSIONS: set = field(default_factory=lambda: {"png", "jpg", "jpeg"})
    TEMP_DIR: Path = field(default_factory=lambda: Path(tempfile.gettempdir()) / "crop_diagnosis")
    
    MAX_QUERY_LENGTH: int = 500
    MAX_RESPONSE_LENGTH: int = 15000
    STREAM_UPDATE_INTERVAL_MS: int = 50
    
    # Theme Colors
    PRIMARY_COLOR: str = "#2ECC71"
    SECONDARY_COLOR: str = "#3498DB"
    ACCENT_COLOR: str = "#E74C3C"
    DARK_BG: str = "#1E1E1E"
    CARD_BG: str = "#2D2D2D"
    TEXT_COLOR: str = "#FFFFFF"
    LIGHT_TEXT: str = "#BDBDBD"
    
    def __post_init__(self):
        self.TEMP_DIR.mkdir(parents=True, exist_ok=True)
        self.OLLAMA_API_URL = f"{self.OLLAMA_HOST}/api/generate"

# Logging Setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('crop_diagnosis.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ============================================================================
# COMPLETE MULTI-LANGUAGE SUPPORT - ALL 11 LANGUAGES
# ============================================================================

LANGUAGES = {
    "English": {
        "code": "en-IN",
        "flag": "🇬🇧",
        "ui": {
            "app_title": "🌾 AI Crop Disease Diagnosis",
            "select_language": "Select Your Language",
            "change_language": "← Change Language",
            "image_preview": "📸 Image Preview",
            "no_image": "No image selected\n\nUpload photo or use camera",
            "upload_btn": "📁 Upload Image\n(PNG/JPG/JPEG only)",
            "camera_btn": "📷 Use Camera",
            "symptom_input": "✍️ Symptom Description (Optional)",
            "placeholder": "Describe symptoms or leave blank...",
            "show_keyboard": "⌨️ Show Keyboard",
            "hide_keyboard": "⌨️ Hide Keyboard",
            "diagnose_btn": "🎯 DIAGNOSE CROP DISEASE",
            "diagnosis_results": "📋 Diagnosis Results",
            "status_ready": "✅ Ready",
            "status_analyzing": "⏳ Analyzing... Please wait",
            "status_camera_active": "🎥 Camera active - Click to capture",
            "analyzing_message": "⏳ Analyzing crop image...\n\nThis may take 30-90 seconds.\n\nPlease wait...",
            "char_count": "{current}/{max}",
            "error_no_image": "No Image",
            "error_no_image_msg": "Please upload an image or use camera first",
            "error_input": "Input Error",
            "error_file": "File Error",
            "error_general": "Error",
            "success_upload": "✅ Image loaded successfully",
            "capture_btn": "📸 CAPTURE PHOTO",
            "zoom_label": "🔍 Text Size:",
            "ollama_info": "Powered by Ollama AI"
        }
    },
    "हिंदी (Hindi)": {
        "code": "hi-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 एआई फसल रोग निदान",
            "select_language": "अपनी भाषा चुनें",
            "change_language": "← भाषा बदलें",
            "image_preview": "📸 छवि पूर्वावलोकन",
            "no_image": "कोई छवि नहीं चुनी गई\n\nफोटो अपलोड करें या कैमरा उपयोग करें",
            "upload_btn": "📁 छवि अपलोड करें\n(केवल PNG/JPG/JPEG)",
            "camera_btn": "📷 कैमरा उपयोग करें",
            "symptom_input": "✍️ लक्षण विवरण (वैकल्पिक)",
            "placeholder": "लक्षण बताएं या खाली छोड़ें...",
            "show_keyboard": "⌨️ कीबोर्ड दिखाएं",
            "hide_keyboard": "⌨️ कीबोर्ड छुपाएं",
            "diagnose_btn": "🎯 फसल रोग का निदान करें",
            "diagnosis_results": "📋 निदान परिणाम",
            "status_ready": "✅ तैयार",
            "status_analyzing": "⏳ विश्लेषण हो रहा है... कृपया प्रतीक्षा करें",
            "status_camera_active": "🎥 कैमरा सक्रिय - फोटो लेने के लिए क्लिक करें",
            "analyzing_message": "⏳ फसल छवि का विश्लेषण हो रहा है...\n\nइसमें 30-90 सेकंड लग सकते हैं।\n\nकृपया प्रतीक्षा करें...",
            "char_count": "{current}/{max}",
            "error_no_image": "कोई छवि नहीं",
            "error_no_image_msg": "कृपया पहले छवि अपलोड करें या कैमरा उपयोग करें",
            "error_input": "इनपुट त्रुटि",
            "error_file": "फाइल त्रुटि",
            "error_general": "त्रुटि",
            "success_upload": "✅ छवि सफलतापूर्वक लोड हुई",
            "capture_btn": "📸 फोटो लें",
            "zoom_label": "🔍 पाठ आकार:",
            "ollama_info": "ओलामा एआई द्वारा संचालित"
        }
    },
    "ગુજરાતી (Gujarati)": {
        "code": "gu-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 એઆઈ પાક રોગ નિદાન",
            "select_language": "તમારી ભાષા પસંદ કરો",
            "change_language": "← ભાષા બદલો",
            "image_preview": "📸 છબી પૂર્વાવલોકન",
            "no_image": "કોઈ છબી પસંદ નથી\n\nફોટો અપલોડ કરો અથવા કેમેરા વાપરો",
            "upload_btn": "📁 છબી અપલોડ કરો\n(માત્ર PNG/JPG/JPEG)",
            "camera_btn": "📷 કેમેરા વાપરો",
            "symptom_input": "✍️ લક્ષણ વર્ણન (વૈકલ્પિક)",
            "placeholder": "લક્ષણો વર્ણવો અથવા ખાલી છોડો...",
            "show_keyboard": "⌨️ કીબોર્ડ બતાવો",
            "hide_keyboard": "⌨️ કીબોર્ડ છુપાવો",
            "diagnose_btn": "🎯 પાક રોગનું નિદાન કરો",
            "diagnosis_results": "📋 નિદાન પરિણામો",
            "status_ready": "✅ તૈયાર",
            "status_analyzing": "⏳ વિશ્લેષણ થઈ રહ્યું છે... કૃપા કરી રાહ જુઓ",
            "status_camera_active": "🎥 કેમેરા સક્રિય - ફોટો લેવા માટે ક્લિક કરો",
            "analyzing_message": "⏳ પાક છબીનું વિશ્લેષણ થઈ રહ્યું છે...\n\nઆમાં 30-90 સેકંડ લાગી શકે છે।\n\nકૃપા કરી રાહ જુઓ...",
            "char_count": "{current}/{max}",
            "error_no_image": "કોઈ છબી નથી",
            "error_no_image_msg": "કૃપા કરીને પહેલા છબી અપલોડ કરો અથવા કેમેરા વાપરો",
            "error_input": "ઇનપુટ ભૂલ",
            "error_file": "ફાઇલ ભૂલ",
            "error_general": "ભૂલ",
            "success_upload": "✅ છબી સફળતાપૂર્વક લોડ થઈ",
            "capture_btn": "📸 ફોટો લો",
            "zoom_label": "🔍 ટેક્સ્ટ સાઈઝ:",
            "ollama_info": "ઓલામા એઆઈ દ્વારા સંચાલિત"
        }
    },
    "मराठी (Marathi)": {
        "code": "mr-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 एआय पीक रोग निदान",
            "select_language": "तुमची भाषा निवडा",
            "change_language": "← भाषा बदला",
            "image_preview": "📸 प्रतिमा पूर्वावलोकन",
            "no_image": "कोणतीही प्रतिमा निवडलेली नाही\n\nफोटो अपलोड करा किंवा कॅमेरा वापरा",
            "upload_btn": "📁 प्रतिमा अपलोड करा\n(फक्त PNG/JPG/JPEG)",
            "camera_btn": "📷 कॅमेरा वापरा",
            "symptom_input": "✍️ लक्षण वर्णन (पर्यायी)",
            "placeholder": "लक्षणे वर्णन करा किंवा रिक्त ठेवा...",
            "show_keyboard": "⌨️ कीबोर्ड दाखवा",
            "hide_keyboard": "⌨️ कीबोर्ड लपवा",
            "diagnose_btn": "🎯 पीक रोग निदान करा",
            "diagnosis_results": "📋 निदान परिणाम",
            "status_ready": "✅ तयार",
            "status_analyzing": "⏳ विश्लेषण होत आहे... कृपया प्रतीक्षा करा",
            "status_camera_active": "🎥 कॅमेरा सक्रिय - फोटो घेण्यासाठी क्लिक करा",
            "analyzing_message": "⏳ पीक प्रतिमेचे विश्लेषण होत आहे...\n\nयास 30-90 सेकंद लागू शकतात।\n\nकृपया प्रतीक्षा करा...",
            "char_count": "{current}/{max}",
            "error_no_image": "प्रतिमा नाही",
            "error_no_image_msg": "कृपया प्रथम प्रतिमा अपलोड करा किंवा कॅमेरा वापरा",
            "error_input": "इनपुट त्रुटी",
            "error_file": "फाइल त्रुटी",
            "error_general": "त्रुटी",
            "success_upload": "✅ प्रतिमा यशस्वीरित्या लोड झाली",
            "capture_btn": "📸 फोटो घ्या",
            "zoom_label": "🔍 मजकूर आकार:",
            "ollama_info": "ओलामा एआय द्वारा संचालित"
        }
    },
    "తెలుగు (Telugu)": {
        "code": "te-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 ఏఐ పంట వ్యాధి నిర్ధారణ",
            "select_language": "మీ భాషను ఎంచుకోండి",
            "change_language": "← భాష మార్చండి",
            "image_preview": "📸 చిత్ర ప్రివ్యూ",
            "no_image": "చిత్రం ఎంచుకోబడలేదు\n\nఫోటో అప్‌లోడ్ చేయండి లేదా కెమెరా ఉపయోగించండి",
            "upload_btn": "📁 చిత్రం అప్‌లోడ్ చేయండి\n(PNG/JPG/JPEG మాత్రమే)",
            "camera_btn": "📷 కెమెరా ఉపయోగించండి",
            "symptom_input": "✍️ లక్షణ వివరణ (ఐచ్ఛికం)",
            "placeholder": "లక్షణాలు వివరించండి లేదా ఖాళీగా వదిలేయండి...",
            "show_keyboard": "⌨️ కీబోర్డ్ చూపించు",
            "hide_keyboard": "⌨️ కీబోర్డ్ దాచు",
            "diagnose_btn": "🎯 పంట వ్యాధిని నిర్ధారించండి",
            "diagnosis_results": "📋 నిర్ధారణ ఫలితాలు",
            "status_ready": "✅ సిద్ధంగా ఉంది",
            "status_analyzing": "⏳ విశ్లేషిస్తోంది... దయచేసి వేచి ఉండండి",
            "status_camera_active": "🎥 కెమెరా చురుకుగా ఉంది - ఫోటో తీయడానికి క్లిక్ చేయండి",
            "analyzing_message": "⏳ పంట చిత్రాన్ని విశ్లేషిస్తోంది...\n\nఇది 30-90 సెకన్లు పట్టవచ్చు।\n\nదయచేసి వేచి ఉండండి...",
            "char_count": "{current}/{max}",
            "error_no_image": "చిత్రం లేదు",
            "error_no_image_msg": "దయచేసి ముందుగా చిత్రాన్ని అప్‌లోడ్ చేయండి లేదా కెమెరా ఉపయోగించండి",
            "error_input": "ఇన్‌పుట్ లోపం",
            "error_file": "ఫైల్ లోపం",
            "error_general": "లోపం",
            "success_upload": "✅ చిత్రం విజయవంతంగా లోడ్ చేయబడింది",
            "capture_btn": "📸 ఫోటో తీయండి",
            "zoom_label": "🔍 టెక్స్ట్ పరిమాణం:",
            "ollama_info": "ఒలామా ఏఐ ద్వారా శక్తివంతం"
        }
    },
    "தமிழ் (Tamil)": {
        "code": "ta-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 AI பயிர் நோய் கண்டறிதல்",
            "select_language": "உங்கள் மொழியைத் தேர்ந்தெடுக்கவும்",
            "change_language": "← மொழியை மாற்று",
            "image_preview": "📸 படம் முன்னோட்டம்",
            "no_image": "படம் தேர்ந்தெடுக்கப்படவில்லை\n\nபடத்தை பதிவேற்றவும் அல்லது கேமராவைப் பயன்படுத்தவும்",
            "upload_btn": "📁 படத்தை பதிவேற்றவும்\n(PNG/JPG/JPEG மட்டும்)",
            "camera_btn": "📷 கேமராவைப் பயன்படுத்து",
            "symptom_input": "✍️ அறிகுறி விளக்கம் (விருப்பத்தேர்வு)",
            "placeholder": "அறிகுறிகளை விவரிக்கவும் அல்லது காலியாக விடவும்...",
            "show_keyboard": "⌨️ விசைப்பலகையைக் காட்டு",
            "hide_keyboard": "⌨️ விசைப்பலகையை மறை",
            "diagnose_btn": "🎯 பயிர் நோயைக் கண்டறி",
            "diagnosis_results": "📋 கண்டறிதல் முடிவுகள்",
            "status_ready": "✅ தயார்",
            "status_analyzing": "⏳ பகுப்பாய்வு செய்கிறது... காத்திருக்கவும்",
            "status_camera_active": "🎥 கேமரா செயலில் - புகைப்படம் எடுக்க கிளிக் செய்யவும்",
            "analyzing_message": "⏳ பயிர் படத்தை பகுப்பாய்வு செய்கிறது...\n\nஇது 30-90 விநாடிகள் எடுக்கலாம்।\n\nதயவுசெய்து காத்திருக்கவும்...",
            "char_count": "{current}/{max}",
            "error_no_image": "படம் இல்லை",
            "error_no_image_msg": "தயவுசெய்து முதலில் படத்தைப் பதிவேற்றவும் அல்லது கேமராவைப் பயன்படுத்தவும்",
            "error_input": "உள்ளீட்டு பிழை",
            "error_file": "கோப்பு பிழை",
            "error_general": "பிழை",
            "success_upload": "✅ படம் வெற்றிகரமாக ஏற்றப்பட்டது",
            "capture_btn": "📸 புகைப்படம் எடு",
            "zoom_label": "🔍 உரை அளவு:",
            "ollama_info": "ஒலாமா ஏஐ மூலம் இயக்கப்படுகிறது"
        }
    },
    "ಕನ್ನಡ (Kannada)": {
        "code": "kn-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 AI ಬೆಳೆ ರೋಗ ನಿರ್ಣಯ",
            "select_language": "ನಿಮ್ಮ ಭಾಷೆಯನ್ನು ಆಯ್ಕೆಮಾಡಿ",
            "change_language": "← ಭಾಷೆ ಬದಲಾಯಿಸಿ",
            "image_preview": "📸 ಚಿತ್ರ ಪೂರ್ವವೀಕ್ಷಣೆ",
            "no_image": "ಯಾವುದೇ ಚಿತ್ರವನ್ನು ಆಯ್ಕೆಮಾಡಿಲ್ಲ\n\nಫೋಟೋ ಅಪ್‌ಲೋಡ್ ಮಾಡಿ ಅಥವಾ ಕ್ಯಾಮರಾ ಬಳಸಿ",
            "upload_btn": "📁 ಚಿತ್ರವನ್ನು ಅಪ್‌ಲೋಡ್ ಮಾಡಿ\n(PNG/JPG/JPEG ಮಾತ್ರ)",
            "camera_btn": "📷 ಕ್ಯಾಮರಾ ಬಳಸಿ",
            "symptom_input": "✍️ ರೋಗಲಕ್ಷಣ ವಿವರಣೆ (ಐಚ್ಛಿಕ)",
            "placeholder": "ರೋಗಲಕ್ಷಣಗಳನ್ನು ವಿವರಿಸಿ ಅಥವಾ ಖಾಲಿ ಬಿಡಿ...",
            "show_keyboard": "⌨️ ಕೀಲಿಮಣೆ ತೋರಿಸಿ",
            "hide_keyboard": "⌨️ ಕೀಲಿಮಣೆ ಮರೆಮಾಡಿ",
            "diagnose_btn": "🎯 ಬೆಳೆ ರೋಗವನ್ನು ನಿರ್ಣಯಿಸಿ",
            "diagnosis_results": "📋 ನಿರ್ಣಯ ಫಲಿತಾಂಶಗಳು",
            "status_ready": "✅ ಸಿದ್ಧವಾಗಿದೆ",
            "status_analyzing": "⏳ ವಿಶ್ಲೇಷಿಸುತ್ತಿದೆ... ದಯವಿಟ್ಟು ನಿರೀಕ್ಷಿಸಿ",
            "status_camera_active": "🎥 ಕ್ಯಾಮರಾ ಸಕ್ರಿಯವಾಗಿದೆ - ಫೋಟೋ ತೆಗೆಯಲು ಕ್ಲಿಕ್ ಮಾಡಿ",
            "analyzing_message": "⏳ ಬೆಳೆ ಚಿತ್ರವನ್ನು ವಿಶ್ಲೇಷಿಸುತ್ತಿದೆ...\n\nಇದು 30-90 ಸೆಕೆಂಡುಗಳನ್ನು ತೆಗೆದುಕೊಳ್ಳಬಹುದು।\n\nದಯವಿಟ್ಟು ನಿರೀಕ್ಷಿಸಿ...",
            "char_count": "{current}/{max}",
            "error_no_image": "ಚಿತ್ರವಿಲ್ಲ",
            "error_no_image_msg": "ದಯವಿಟ್ಟು ಮೊದಲು ಚಿತ್ರವನ್ನು ಅಪ್‌ಲೋಡ್ ಮಾಡಿ ಅಥವಾ ಕ್ಯಾಮರಾ ಬಳಸಿ",
            "error_input": "ಇನ್‌ಪುಟ್ ದೋಷ",
            "error_file": "ಫೈಲ್ ದೋಷ",
            "error_general": "ದೋಷ",
            "success_upload": "✅ ಚಿತ್ರವನ್ನು ಯಶಸ್ವಿಯಾಗಿ ಲೋಡ್ ಮಾಡಲಾಗಿದೆ",
            "capture_btn": "📸 ಫೋಟೋ ತೆಗೆಯಿರಿ",
            "zoom_label": "🔍 ಪಠ್ಯ ಗಾತ್ರ:",
            "ollama_info": "ಒಲಾಮಾ ಎಐ ಮೂಲಕ ಚಾಲಿತ"
        }
    },
    "മലയാളം (Malayalam)": {
        "code": "ml-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 AI വിള രോഗനിർണയം",
            "select_language": "നിങ്ങളുടെ ഭാഷ തിരഞ്ഞെടുക്കുക",
            "change_language": "← ഭാഷ മാറ്റുക",
            "image_preview": "📸 ചിത്ര പ്രിവ്യൂ",
            "no_image": "ചിത്രം തിരഞ്ഞെടുത്തിട്ടില്ല\n\nഫോട്ടോ അപ്‌ലോഡ് ചെയ്യുക അല്ലെങ്കിൽ ക്യാമറ ഉപയോഗിക്കുക",
            "upload_btn": "📁 ചിത്രം അപ്‌ലോഡ് ചെയ്യുക\n(PNG/JPG/JPEG മാത്രം)",
            "camera_btn": "📷 ക്യാമറ ഉപയോഗിക്കുക",
            "symptom_input": "✍️ രോഗലക്ഷണ വിവരണം (ഓപ്ഷണൽ)",
            "placeholder": "രോഗലക്ഷണങ്ഗൾ വിവരിക്കുക അല്ലെങ്കിൽ ശൂന്യമായി വിടുക...",
            "show_keyboard": "⌨️ കീബോർഡ് കാണിക്കുക",
            "hide_keyboard": "⌨️ കീബോർഡ് മറയ്ക്കുക",
            "diagnose_btn": "🎯 വിള രോഗം നിർണയിക്കുക",
            "diagnosis_results": "📋 രോഗനിർണയ ഫലങ്ങൾ",
            "status_ready": "✅ തയ്യാറാണ്",
            "status_analyzing": "⏳ വിശകലനം ചെയ്യുന്നു... കാത്തിരിക്കുക",
            "status_camera_active": "🎥 ക്യാമറ സജീവമാണ് - ഫോട്ടോ എടുക്കാൻ ക്ലിക്ക് ചെയ്യുക",
            "analyzing_message": "⏳ വിള ചിത്രം വിശകലനം ചെയ്യുന്നു...\n\nഇതിന് 30-90 സെക്കൻഡ് എടുത്തേക്കാം।\n\nദയവായി കാത്തിരിക്കുക...",
            "char_count": "{current}/{max}",
            "error_no_image": "ചിത്രമില്ല",
            "error_no_image_msg": "ദയവായി ആദ്യം ഒരു ചിത്രം അപ്‌ലോഡ് ചെയ്യുക അല്ലെങ്കിൽ ക്യാമറ ഉപയോഗിക്കുക",
            "error_input": "ഇൻപുട്ട് പിശക്",
            "error_file": "ഫയൽ പിശക്",
            "error_general": "പിശക്",
            "success_upload": "✅ ചിത്രം വിജയകരമായി ലോഡ് ചെയ്തു",
            "capture_btn": "📸 ഫോട്ടോ എടുക്കുക",
            "zoom_label": "🔍 വാചക വലുപ്പം:",
            "ollama_info": "ഒലാമ എഐ വഴി പ്രവർത്തിക്കുന്നു"
        }
    },
    "বাংলা (Bengali)": {
        "code": "bn-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 AI ফসল রোগ নির্ণয়",
            "select_language": "আপনার ভাষা নির্বাচন করুন",
            "change_language": "← ভাষা পরিবর্তন করুন",
            "image_preview": "📸 ছবির প্রিভিউ",
            "no_image": "কোনো ছবি নির্বাচিত নেই\n\nছবি আপলোড করুন বা ক্যামেরা ব্যবহার করুন",
            "upload_btn": "📁 ছবি আপলোড করুন\n(শুধুমাত্র PNG/JPG/JPEG)",
            "camera_btn": "📷 ক্যামেরা ব্যবহার করুন",
            "symptom_input": "✍️ লক্ষণ বর্ণনা (ঐচ্ছিক)",
            "placeholder": "লক্ষণ বর্ণনা করুন বা খালি রাখুন...",
            "show_keyboard": "⌨️ কীবোর্ড দেখান",
            "hide_keyboard": "⌨️ কীবোর্ড লুকান",
            "diagnose_btn": "🎯 ফসলের রোগ নির্ণয় করুন",
            "diagnosis_results": "📋 নির্ণয়ের ফলাফল",
            "status_ready": "✅ প্রস্তুত",
            "status_analyzing": "⏳ বিশ্লেষণ করা হচ্ছে... অপেক্ষা করুন",
            "status_camera_active": "🎥 ক্যামেরা সক্রিয় - ছবি তুলতে ক্লিক করুন",
            "analyzing_message": "⏳ ফসলের ছবি বিশ্লেষণ করা হচ্ছে...\n\nএটি 30-90 সেকেন্ড সময় নিতে পারে।\n\nঅনুগ্রহ করে অপেক্ষা করুন...",
            "char_count": "{current}/{max}",
            "error_no_image": "কোনো ছবি নেই",
            "error_no_image_msg": "অনুগ্রহ করে প্রথমে একটি ছবি আপলোড করুন বা ক্যামেরা ব্যবহার করুন",
            "error_input": "ইনপুট ত্রুটি",
            "error_file": "ফাইল ত্রুটি",
            "error_general": "ত্রুটি",
            "success_upload": "✅ ছবি সফলভাবে লোড হয়েছে",
            "capture_btn": "📸 ছবি তুলুন",
            "zoom_label": "🔍 টেক্সট সাইজ:",
            "ollama_info": "ওলামা এআই দ্বারা চালিত"
        }
    },
    "ਪੰਜਾਬੀ (Punjabi)": {
        "code": "pa-IN",
        "flag": "🇮🇳",
        "ui": {
            "app_title": "🌾 AI ਫਸਲ ਰੋਗ ਨਿਦਾਨ",
            "select_language": "ਆਪਣੀ ਭਾਸ਼ਾ ਚੁਣੋ",
            "change_language": "← ਭਾਸ਼ਾ ਬਦਲੋ",
            "image_preview": "📸 ਤਸਵੀਰ ਪੂਰਵਦਰਸ਼ਨ",
            "no_image": "ਕੋਈ ਤਸਵੀਰ ਨਹੀਂ ਚੁਣੀ\n\nਫੋਟੋ ਅੱਪਲੋਡ ਕਰੋ ਜਾਂ ਕੈਮਰਾ ਵਰਤੋ",
            "upload_btn": "📁 ਤਸਵੀਰ ਅੱਪਲੋਡ ਕਰੋ\n(ਸਿਰਫ PNG/JPG/JPEG)",
            "camera_btn": "📷 ਕੈਮਰਾ ਵਰਤੋ",
            "symptom_input": "✍️ ਲੱਛਣ ਵੇਰਵਾ (ਵਿਕਲਪਿਕ)",
            "placeholder": "ਲੱਛਣਾਂ ਦਾ ਵਰਣਨ ਕਰੋ ਜਾਂ ਖਾਲੀ ਛੱਡੋ...",
            "show_keyboard": "⌨️ ਕੀਬੋਰਡ ਦਿਖਾਓ",
            "hide_keyboard": "⌨️ ਕੀਬੋਰਡ ਲੁਕਾਓ",
            "diagnose_btn": "🎯 ਫਸਲ ਰੋਗ ਦਾ ਨਿਦਾਨ ਕਰੋ",
            "diagnosis_results": "📋 ਨਿਦਾਨ ਨਤੀਜੇ",
            "status_ready": "✅ ਤਿਆਰ",
            "status_analyzing": "⏳ ਵਿਸ਼ਲੇਸ਼ਣ ਹੋ ਰਿਹਾ ਹੈ... ਕਿਰਪਾ ਕਰਕੇ ਉਡੀਕ ਕਰੋ",
            "status_camera_active": "🎥 ਕੈਮਰਾ ਸਰਗਰਮ - ਫੋਟੋ ਲੈਣ ਲਈ ਕਲਿੱਕ ਕਰੋ",
            "analyzing_message": "⏳ ਫਸਲ ਦੀ ਤਸਵੀਰ ਦਾ ਵਿਸ਼ਲੇਸ਼ਣ ਹੋ ਰਿਹਾ ਹੈ...\n\nਇਸ ਵਿੱਚ 30-90 ਸਕਿੰਟ ਲੱਗ ਸਕਦੇ ਹਨ।\n\nਕਿਰਪਾ ਕਰਕੇ ਉਡੀਕ ਕਰੋ...",
            "char_count": "{current}/{max}",
            "error_no_image": "ਕੋਈ ਤਸਵੀਰ ਨਹੀਂ",
            "error_no_image_msg": "ਕਿਰਪਾ ਕਰਕੇ ਪਹਿਲਾਂ ਤਸਵੀਰ ਅੱਪਲੋਡ ਕਰੋ ਜਾਂ ਕੈਮਰਾ ਵਰਤੋ",
            "error_input": "ਇਨਪੁੱਟ ਗਲਤੀ",
            "error_file": "ਫਾਈਲ ਗਲਤੀ",
            "error_general": "ਗਲਤੀ",
            "success_upload": "✅ ਤਸਵੀਰ ਸਫਲਤਾਪੂਰਵਕ ਲੋਡ ਹੋਈ",
            "capture_btn": "📸 ਫੋਟੋ ਲਓ",
            "zoom_label": "🔍 ਟੈਕਸਟ ਸਾਈਜ਼:",
            "ollama_info": "ਓਲਾਮਾ ਏਆਈ ਦੁਆਰਾ ਸੰਚਾਲਿਤ"
        }
    },
    "اردو (Urdu)": {
        "code": "ur-PK",
        "flag": "🇵🇰",
        "ui": {
            "app_title": "🌾 AI فصل کی بیماری کی تشخیص",
            "select_language": "اپنی زبان منتخب کریں",
            "change_language": "← زبان تبدیل کریں",
            "image_preview": "📸 تصویر کا پیش منظر",
            "no_image": "کوئی تصویر منتخب نہیں\n\nتصویر اپ لوڈ کریں یا کیمرہ استعمال کریں",
            "upload_btn": "📁 تصویر اپ لوڈ کریں\n(صرف PNG/JPG/JPEG)",
            "camera_btn": "📷 کیمرہ استعمال کریں",
            "symptom_input": "✍️ علامات کی تفصیل (اختیاری)",
            "placeholder": "علامات بیان کریں یا خالی چھوڑیں...",
            "show_keyboard": "⌨️ کی بورڈ دکھائیں",
            "hide_keyboard": "⌨️ کی بورڈ چھپائیں",
            "diagnose_btn": "🎯 فصل کی بیماری کی تشخیص کریں",
            "diagnosis_results": "📋 تشخیص کے نتائج",
            "status_ready": "✅ تیار",
            "status_analyzing": "⏳ تجزیہ ہو رہا ہے... براہ کرم انتظار کریں",
            "status_camera_active": "🎥 کیمرہ فعال - تصویر لینے کے لیے کلک کریں",
            "analyzing_message": "⏳ فصل کی تصویر کا تجزیہ ہو رہا ہے...\n\nاس میں 30-90 سیکنڈ لگ سکتے ہیں۔\n\nبراہ کرم انتظار کریں...",
            "char_count": "{current}/{max}",
            "error_no_image": "کوئی تصویر نہیں",
            "error_no_image_msg": "براہ کرم پہلے تصویر اپ لوڈ کریں یا کیمرہ استعمال کریں",
            "error_input": "ان پٹ کی خرابی",
            "error_file": "فائل کی خرابی",
            "error_general": "خرابی",
            "success_upload": "✅ تصویر کامیابی سے لوڈ ہوئی",
            "capture_btn": "📸 تصویر لیں",
            "zoom_label": "🔍 متن کا سائز:",
            "ollama_info": "اولاما اے آئی سے چلایا گیا"
        }
    }
}

# Complete Native Keyboards for ALL Languages
KEYBOARDS = {
    "English": [
        "Q W E R T Y U I O P",
        "A S D F G H J K L",
        "Z X C V B N M",
        ", . ? ! ␣ ⌫"
    ],
    "हिंदी (Hindi)": [
        "अ आ इ ई उ ऊ ए ऐ ओ औ",
        "क ख ग घ ङ च छ ज झ ञ",
        "ट ठ ड ढ ण त थ द ध न",
        "प फ ब भ म य र ल व",
        "श ष स ह ा ि ी ु ू",
        "े ै ो ौ ं ः ् ␣ ⌫"
    ],
    "ગુજરાતી (Gujarati)": [
        "અ આ ઇ ઈ ઉ ઊ એ ઐ ઓ ઔ",
        "ક ખ ગ ઘ ઙ ચ છ જ ઝ ઞ",
        "ટ ઠ ડ ઢ ણ ત થ દ ધ ન",
        "પ ફ બ ભ મ ય ર લ વ",
        "શ ષ સ હ ા િ ી ુ ૂ",
        "ે ૈ ો ૌ ં ઃ ્ ␣ ⌫"
    ],
    "मराठी (Marathi)": [
        "अ आ इ ई उ ऊ ए ऐ ओ औ",
        "क ख ग घ ङ च छ ज झ ञ",
        "ट ठ ड ढ ण त थ द ध न",
        "प फ ब भ म य र ल व",
        "श ष स ह ा ि ी ु ू",
        "े ै ो ौ ं ः ् ␣ ⌫"
    ],
    "తెలుగు (Telugu)": [
        "అ ఆ ఇ ఈ ఉ ఊ ఎ ఏ ఒ ఓ",
        "క ఖ గ ఘ ఙ చ ఛ జ ఝ ఞ",
        "ట ఠ డ ఢ ణ త థ ద ధ న",
        "ప ఫ బ భ మ య ర ల వ",
        "శ ష స హ ా ి ీ ు ూ",
        "ె ే ై ొ ో ౌ ం ః ్ ␣ ⌫"
    ],
    "தமிழ் (Tamil)": [
        "அ ஆ இ ஈ உ ஊ எ ஏ ஒ ஓ",
        "க ங ச ஞ ட ண த ந ப ம",
        "ய ர ல வ ழ ள ற ன",
        "ா ி ீ ு ூ ெ ே ை ொ ோ ௌ ் ␣ ⌫"
    ],
    "ಕನ್ನಡ (Kannada)": [
        "ಅ ಆ ಇ ಈ ಉ ಊ ಎ ಏ ಒ ಓ",
        "ಕ ಖ ಗ ಘ ಙ ಚ ಛ ಜ ಝ ಞ",
        "ಟ ಠ ಡ ಢ ಣ ತ ಥ ದ ಧ ನ",
        "ಪ ಫ ಬ ಭ ಮ ಯ ರ ಲ ವ",
        "ಶ ಷ ಸ ಹ ಾ ಿ ೀ ು ೂ",
        "ೆ ೇ ೈ ೊ ೋ ೌ ಂ ಃ ್ ␣ ⌫"
    ],
    "മലയാളം (Malayalam)": [
        "അ ആ ഇ ഈ ഉ ഊ എ ഏ ഒ ഓ",
        "ക ഖ ഗ ഘ ങ ച ഛ ജ ഝ ഞ",
        "ട ഠ ഡ ഢ ണ ത ഥ ദ ധ ന",
        "പ ഫ ബ ഭ മ യ ര ല വ",
        "ശ ഷ സ ഹ ാ ി ീ ു ൂ",
        "െ േ ൈ ൊ ോ ൌ ം ഃ ് ␣ ⌫"
    ],
    "বাংলা (Bengali)": [
        "অ আ ই ঈ উ ঊ এ ঐ ও ঔ",
        "ক খ গ ঘ ঙ চ ছ জ ঝ ঞ",
        "ট ঠ ড ঢ ণ ত থ দ ধ ন",
        "প ফ ব ভ ম য র ল শ",
        "ষ স হ া ি ী ু ূ",
        "ে ৈ ো ৌ ং ঃ ্ ␣ ⌫"
    ],
    "ਪੰਜਾਬੀ (Punjabi)": [
        "ਅ ਆ ਇ ਈ ਉ ਊ ਏ ਐ ਓ ਔ",
        "ਕ ਖ ਗ ਘ ਙ ਚ ਛ ਜ ਝ ਞ",
        "ਟ ਠ ਡ ਢ ਣ ਤ ਥ ਦ ਧ ਨ",
        "ਪ ਫ ਬ ਭ ਮ ਯ ਰ ਲ ਵ",
        "ਸ਼ ਸ ਹ ਾ ਿ ੀ ੁ ੂ",
        "ੇ ੈ ੋ ੌ ਂ ਃ ੍ ␣ ⌫"
    ],
    "اردو (Urdu)": [
        "ا آ ب پ ت ٹ ث ج چ ح",
        "خ د ڈ ذ ر ڑ ز ژ س ش",
        "ص ض ط ظ ع غ ف ق ک گ",
        "ل م ن ں و ہ ء ی ے ␣ ⌫"
    ]
}

# ============================================================================
# EXCEPTIONS
# ============================================================================

class CropDiagnosisException(Exception):
    pass

class FileValidationError(CropDiagnosisException):
    pass

class APIConnectionError(CropDiagnosisException):
    pass

class InputValidationError(CropDiagnosisException):
    pass

# ============================================================================
# FILE VALIDATOR
# ============================================================================

class FileValidator:
    def __init__(self, config: AppConfig):
        self.config = config
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def validate_file(self, file_path: str) -> Path:
        try:
            file_path = Path(file_path).resolve()
            
            if not file_path.exists():
                raise FileValidationError(f"File does not exist: {file_path}")
            
            extension = file_path.suffix.lower().lstrip('.')
            if extension not in self.config.ALLOWED_EXTENSIONS:
                allowed = ", ".join(self.config.ALLOWED_EXTENSIONS).upper()
                raise FileValidationError(f"Invalid file type: {extension.upper()}\nAllowed: {allowed}")
            
            file_size_mb = file_path.stat().st_size / (1024 * 1024)
            if file_size_mb > self.config.MAX_FILE_SIZE_MB:
                raise FileValidationError(f"File too large: {file_size_mb:.2f}MB (max: {self.config.MAX_FILE_SIZE_MB}MB)")
            
            if not self._verify_image_magic_bytes(file_path):
                raise FileValidationError("File is not a valid image")
            
            self.logger.info(f"✅ File validated: {file_path}")
            return file_path
            
        except FileValidationError:
            raise
        except Exception as e:
            raise FileValidationError(f"Validation failed: {str(e)}")
    
    @staticmethod
    def _verify_image_magic_bytes(file_path: Path) -> bool:
        magic_bytes = {b'\x89PNG': 'png', b'\xff\xd8\xff': 'jpg'}
        try:
            with open(file_path, 'rb') as f:
                file_start = f.read(10)
                return any(file_start.startswith(magic) for magic in magic_bytes.keys())
        except:
            return False

# ============================================================================
# INPUT VALIDATOR
# ============================================================================

class InputValidator:
    def __init__(self, config: AppConfig):
        self.config = config
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def validate_query(self, query: str) -> str:
        if not isinstance(query, str):
            raise InputValidationError("Query must be a string")
        
        query = query.strip()
        
        if len(query) > self.config.MAX_QUERY_LENGTH:
            raise InputValidationError(f"Query too long: {len(query)} chars (max: {self.config.MAX_QUERY_LENGTH})")
        
        dangerous_patterns = ['<script', 'javascript:', 'onerror=', 'onclick=']
        if any(pattern in query.lower() for pattern in dangerous_patterns):
            raise InputValidationError("Query contains invalid patterns")
        
        self.logger.info(f"✅ Query validated: {len(query)} chars")
        return query

# ============================================================================
# API CLIENT - MULTI-LANGUAGE ACCURATE IMAGE RECOGNITION
# ============================================================================

class APIClient:
    def __init__(self, config: AppConfig):
        self.config = config
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def analyze_image(self, image_path: Path, query: str, language: str) -> str:
        """ACCURATE Multi-Language Image Analysis"""
        try:
            self.logger.info(f"📸 Processing: {image_path}")
            
            # Optimize image
            img = Image.open(image_path)
            max_size = 768
            if img.width > max_size or img.height > max_size:
                img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
                temp_path = self.config.TEMP_DIR / f"resized_{image_path.name}"
                img.save(temp_path, quality=85, optimize=True)
                image_path = temp_path
                self.logger.info(f"✅ Resized to {img.size}")
            
            # Encode
            with open(image_path, "rb") as f:
                image_b64 = base64.b64encode(f.read()).decode('utf-8')
            
            # Build LANGUAGE-SPECIFIC prompt
            prompt = self._build_multilang_prompt(language, query)
            
            # API Call
            payload = {
                "model": self.config.OLLAMA_MODEL,
                "prompt": prompt,
                "images": [image_b64],
                "stream": False,
                "options": {
                    "temperature": 0.2,
                    "num_predict": 1000
                }
            }
            
            self.logger.info(f"🌐 Calling Ollama API in {language}...")
            
            response = requests.post(
                self.config.OLLAMA_API_URL,
                json=payload,
                timeout=self.config.OLLAMA_TIMEOUT
            )
            
            if response.status_code == 200:
                data = response.json()
                result = data.get('response', '').strip()
                
                if not result or len(result) < 20:
                    raise APIConnectionError("Empty response from AI. Please try again.")
                
                self.logger.info(f"✅ Success - {len(result)} chars in {language}")
                return result
            else:
                raise APIConnectionError(f"API Error {response.status_code}")
        
        except requests.exceptions.Timeout:
            msg = "⏱️ Timeout\n\nSolutions:\n1. Use smaller image\n2. Restart Ollama\n3. Check resources"
            raise APIConnectionError(msg)
        except requests.exceptions.ConnectionError:
            msg = f"🔌 Cannot connect\n\nRun: ollama serve\nInstall: ollama pull {self.config.OLLAMA_MODEL}"
            raise APIConnectionError(msg)
        except Exception as e:
            raise APIConnectionError(f"Error: {str(e)}")
    
    def _build_multilang_prompt(self, language: str, symptoms: str) -> str:
        """Build language-specific prompt for ACCURATE recognition"""
        
        lang_code = LANGUAGES.get(language, {}).get("code", "en-IN")
        
        # Get language name for instruction
        lang_instruction = {
            "English": "English",
            "हिंदी (Hindi)": "Hindi (हिंदी)",
            "ગુજરાતી (Gujarati)": "Gujarati (ગુજરાતી)",
            "मराठी (Marathi)": "Marathi (मराठी)",
            "తెలుగు (Telugu)": "Telugu (తెలుగు)",
            "தமிழ் (Tamil)": "Tamil (தமிழ்)",
            "ಕನ್ನಡ (Kannada)": "Kannada (ಕನ್ನಡ)",
            "മലയാളം (Malayalam)": "Malayalam (മലയാളം)",
            "বাংলা (Bengali)": "Bengali (বাংলা)",
            "ਪੰਜਾਬੀ (Punjabi)": "Punjabi (ਪੰਜਾਬੀ)",
            "اردو (Urdu)": "Urdu (اردو)"
        }.get(language, "English")
        
        prompt = f"""You are an agricultural expert analyzing a crop disease image.

USER SYMPTOMS: {symptoms if symptoms else 'Analyze what you see'}

ANALYZE THE IMAGE AND PROVIDE DIAGNOSIS IN {lang_instruction} LANGUAGE.

FORMAT YOUR RESPONSE AS FOLLOWS:

📋 रोग/ਰੋਗ/રોગ/DISEASE: [Disease name in {lang_instruction}]

🔍 लक्षण/ਲੱਛਣ/લક્ષણો/SYMPTOMS:
- [Visual symptom 1 in {lang_instruction}]
- [Visual symptom 2 in {lang_instruction}]

🌾 फसल/ਫਸਲ/પાક/CROP: [Crop type in {lang_instruction}]

💊 उपचार/ਇਲਾਜ/સારવાર/TREATMENT:
1. [Treatment step 1 in {lang_instruction}]
2. [Treatment step 2 in {lang_instruction}]

⚠️ सलाह/ਸਲਾਹ/સલાહ/ADVICE: [When to seek help in {lang_instruction}]

IMPORTANT: 
- Write ENTIRELY in {lang_instruction} script/language
- Be specific about the disease
- Provide actionable treatment steps
- Keep response under 600 words"""

        return prompt

# ============================================================================
# CAMERA MANAGER
# ============================================================================

class CameraManager:
    def __init__(self, config: AppConfig):
        self.config = config
        self.logger = logging.getLogger(self.__class__.__name__)
        self.cap = None
        self._lock = Lock()
    
    def start(self) -> bool:
        with self._lock:
            if self.cap is None:
                try:
                    self.cap = cv2.VideoCapture(0)
                    if not self.cap.isOpened():
                        return False
                    self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
                    self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
                    self.logger.info("✅ Camera started")
                    return True
                except Exception as e:
                    self.logger.error(f"Camera error: {e}")
                    return False
            return True
    
    def get_frame(self) -> Optional[tuple]:
        with self._lock:
            if self.cap is None:
                return None
            try:
                ret, frame = self.cap.read()
                return (ret, frame) if ret else None
            except:
                return None
    
    def stop(self):
        with self._lock:
            if self.cap:
                try:
                    self.cap.release()
                    self.logger.info("✅ Camera stopped")
                except:
                    pass
                finally:
                    self.cap = None

# ============================================================================
# MAIN APPLICATION - COMPLETE MULTI-LANGUAGE UI
# ============================================================================

class CropDiagnosisApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.config = AppConfig()
        self.logger = logging.getLogger(self.__class__.__name__)
        
        # Components
        self.file_validator = FileValidator(self.config)
        self.input_validator = InputValidator(self.config)
        self.api_client = APIClient(self.config)
        self.camera_manager = CameraManager(self.config)
        
        # State
        self.current_language = "English"
        self.current_image_path: Optional[Path] = None
        self.is_analyzing = False
        self.camera_active = False
        self.current_font_size = 11
        
        # Setup
        self._setup_theme()
        self._setup_language_selection()
        
        self.logger.info("🌾 App initialized")
    
    def _setup_theme(self):
        ctk.set_appearance_mode("dark")
        ctk.set_default_color_theme("green")
        self.geometry("1400x900")
        self.title("🌾 AI Crop Disease Diagnosis")
        self.configure(fg_color=self.config.DARK_BG)
    
    def _get_ui_text(self, key: str) -> str:
        """Get localized UI text"""
        return LANGUAGES[self.current_language]["ui"].get(key, key)
    
    def _setup_language_selection(self):
        self._clear_screen()
        
        main_frame = ctk.CTkFrame(self, fg_color=self.config.DARK_BG)
        main_frame.pack(fill="both", expand=True, padx=20, pady=20)
        
        header = ctk.CTkLabel(main_frame, text="🌾 AI Crop Disease Diagnosis",
                             font=("Segoe UI", 36, "bold"), text_color=self.config.PRIMARY_COLOR)
        header.pack(pady=30)
        
        subtitle = ctk.CTkLabel(main_frame, 
                               text="Select Your Language | भाषा चुनें | ભાષા પસંદ કરો",
                               font=("Segoe UI", 14), text_color=self.config.LIGHT_TEXT)
        subtitle.pack(pady=(0, 40))
        
        grid_frame = ctk.CTkFrame(main_frame, fg_color="transparent")
        grid_frame.pack(padx=20, pady=20)
        
        for idx, lang_name in enumerate(LANGUAGES.keys()):
            lang_info = LANGUAGES[lang_name]
            btn = ctk.CTkButton(grid_frame, text=f"{lang_info['flag']} {lang_name}",
                              font=("Segoe UI", 13, "bold"), width=220, height=55,
                              fg_color=self.config.SECONDARY_COLOR,
                              hover_color=self.config.PRIMARY_COLOR,
                              command=lambda l=lang_name: self._launch_main(l))
            btn.grid(row=idx//3, column=idx%3, padx=12, pady=12)
    
    def _launch_main(self, language: str):
        self.current_language = language
        self.logger.info(f"🌐 Language: {language}")
        self._clear_screen()
        self._setup_main_ui()
    
    def _setup_main_ui(self):
        """Setup main UI - ALL elements in selected language"""
        main_frame = ctk.CTkFrame(self, fg_color=self.config.DARK_BG)
        main_frame.pack(fill="both", expand=True, padx=10, pady=10)
        
        # Header
        header_frame = ctk.CTkFrame(main_frame, fg_color="transparent")
        header_frame.pack(fill="x", padx=20, pady=10)
        
        # TRANSLATED back button
        back_btn = ctk.CTkButton(header_frame, text=self._get_ui_text("change_language"),
                                font=("Segoe UI", 10), width=150, height=35,
                                fg_color=self.config.ACCENT_COLOR,
                                command=self._setup_language_selection)
        back_btn.pack(side="left")
        
        title = ctk.CTkLabel(header_frame, text=f"{self._get_ui_text('app_title')}",
                            font=("Segoe UI", 22, "bold"), text_color=self.config.PRIMARY_COLOR)
        title.pack(side="left", padx=20)
        
        # Content
        content_frame = ctk.CTkFrame(main_frame, fg_color="transparent")
        content_frame.pack(fill="both", expand=True, padx=10, pady=10)
        
        # Left - Image
        left_frame = ctk.CTkFrame(content_frame, fg_color=self.config.CARD_BG, corner_radius=15)
        left_frame.pack(side="left", fill="both", expand=True, padx=(0, 10), pady=10)
        self._setup_image_area(left_frame)
        
        # Right - Input/Results
        right_frame = ctk.CTkFrame(content_frame, fg_color=self.config.CARD_BG, corner_radius=15)
        right_frame.pack(side="right", fill="both", expand=True, padx=(10, 0), pady=10)
        self._setup_input_area(right_frame)
    
    def _setup_image_area(self, parent):
        """Image area - TRANSLATED"""
        title = ctk.CTkLabel(parent, text=self._get_ui_text("image_preview"),
                            font=("Segoe UI", 16, "bold"), text_color=self.config.PRIMARY_COLOR)
        title.pack(padx=15, pady=(15, 10))
        
        self.image_label = ctk.CTkLabel(parent, text=self._get_ui_text("no_image"),
                                       font=("Segoe UI", 12), text_color=self.config.LIGHT_TEXT,
                                       fg_color=self.config.DARK_BG, width=500, height=500, corner_radius=10)
        self.image_label.pack(padx=15, pady=10, fill="both", expand=True)
        
        button_frame = ctk.CTkFrame(parent, fg_color="transparent")
        button_frame.pack(padx=15, pady=15, fill="x")
        
        # TRANSLATED buttons
        upload_btn = ctk.CTkButton(button_frame, text=self._get_ui_text("upload_btn"),
                                  font=("Segoe UI", 12, "bold"), height=55,
                                  fg_color=self.config.PRIMARY_COLOR, hover_color="#27AE60",
                                  command=self._upload_file)
        upload_btn.pack(side="left", padx=5, fill="x", expand=True)
        
        self.camera_btn = ctk.CTkButton(button_frame, text=self._get_ui_text("camera_btn"),
                                       font=("Segoe UI", 12, "bold"), height=55,
                                       fg_color=self.config.SECONDARY_COLOR, hover_color="#2980B9",
                                       command=self._toggle_camera)
        self.camera_btn.pack(side="left", padx=5, fill="x", expand=True)
    
    def _setup_input_area(self, parent):
        """Input area - TRANSLATED"""
        input_title = ctk.CTkLabel(parent, text=self._get_ui_text("symptom_input"),
                                   font=("Segoe UI", 14, "bold"), text_color=self.config.PRIMARY_COLOR)
        input_title.pack(padx=15, pady=(15, 10))
        
        text_frame = ctk.CTkFrame(parent, fg_color=self.config.DARK_BG, corner_radius=10)
        text_frame.pack(padx=15, pady=10, fill="x")
        
        self.query_entry = ctk.CTkEntry(text_frame, placeholder_text=self._get_ui_text("placeholder"),
                                       font=("Segoe UI", 12), height=45)
        self.query_entry.pack(padx=10, pady=10, fill="x")
        self.query_entry.bind("<KeyRelease>", self._update_char_count)
        
        self.char_counter = ctk.CTkLabel(text_frame, text=self._get_ui_text("char_count").format(current=0, max=500),
                                        font=("Segoe UI", 9), text_color=self.config.LIGHT_TEXT)
        self.char_counter.pack(padx=10, pady=(0, 10))
        
        # TRANSLATED keyboard button
        self.keyboard_btn = ctk.CTkButton(text_frame, text=self._get_ui_text("show_keyboard"),
                                         font=("Segoe UI", 10), height=38, fg_color="#34495E",
                                         command=self._toggle_keyboard)
        self.keyboard_btn.pack(padx=10, pady=(0, 10), fill="x")
        self.keyboard_visible = False
        self.keyboard_frame = ctk.CTkFrame(text_frame, fg_color=self.config.DARK_BG, corner_radius=8)
        
        # TRANSLATED diagnose button
        self.diagnose_btn = ctk.CTkButton(parent, text=self._get_ui_text("diagnose_btn"),
                                         font=("Segoe UI", 15, "bold"), height=65,
                                         fg_color=self.config.PRIMARY_COLOR, hover_color="#27AE60",
                                         command=self._diagnose)
        self.diagnose_btn.pack(padx=15, pady=15, fill="x")
        
        # Results with ZOOM
        results_header = ctk.CTkFrame(parent, fg_color="transparent")
        results_header.pack(padx=15, pady=(10, 5), fill="x")
        
        results_title = ctk.CTkLabel(results_header, text=self._get_ui_text("diagnosis_results"),
                                     font=("Segoe UI", 13, "bold"), text_color=self.config.PRIMARY_COLOR)
        results_title.pack(side="left")
        
        # ZOOM SLIDER - TRANSLATED
        zoom_frame = ctk.CTkFrame(results_header, fg_color="transparent")
        zoom_frame.pack(side="right")
        
        zoom_label = ctk.CTkLabel(zoom_frame, text=self._get_ui_text("zoom_label"),
                                 font=("Segoe UI", 10), text_color=self.config.LIGHT_TEXT)
        zoom_label.pack(side="left", padx=(0, 5))
        
        self.zoom_slider = ctk.CTkSlider(zoom_frame, from_=8, to=20, number_of_steps=12,
                                        width=120, command=self._update_font_size)
        self.zoom_slider.set(11)
        self.zoom_slider.pack(side="left")
        
        self.zoom_value_label = ctk.CTkLabel(zoom_frame, text="11", font=("Segoe UI", 9),
                                            text_color=self.config.LIGHT_TEXT, width=25)
        self.zoom_value_label.pack(side="left", padx=(5, 0))
        
        # Results textbox
        self.output_textbox = ctk.CTkTextbox(parent, font=("Segoe UI", self.current_font_size),
                                            text_color=self.config.TEXT_COLOR,
                                            fg_color=self.config.DARK_BG, wrap="word")
        self.output_textbox.pack(padx=15, pady=(5, 15), fill="both", expand=True)
        
        # TRANSLATED status
        self.status_label = ctk.CTkLabel(parent, text=self._get_ui_text("status_ready"),
                                        font=("Segoe UI", 10), text_color=self.config.PRIMARY_COLOR)
        self.status_label.pack(padx=15, pady=(0, 10))
    
    def _update_font_size(self, value):
        """Update text size"""
        self.current_font_size = int(value)
        self.zoom_value_label.configure(text=str(self.current_font_size))
        self.output_textbox.configure(font=("Segoe UI", self.current_font_size))
    
    def _toggle_keyboard(self):
        """Toggle keyboard - shows LANGUAGE-SPECIFIC keyboard"""
        if self.keyboard_visible:
            self.keyboard_frame.pack_forget()
            self.keyboard_btn.configure(text=self._get_ui_text("show_keyboard"))
            self.keyboard_visible = False
        else:
            self.keyboard_frame.pack(padx=10, pady=10, fill="both")
            self._populate_keyboard()
            self.keyboard_btn.configure(text=self._get_ui_text("hide_keyboard"))
            self.keyboard_visible = True
    
    def _populate_keyboard(self):
        """Populate with LANGUAGE-SPECIFIC characters"""
        for widget in self.keyboard_frame.winfo_children():
            widget.destroy()
        
        # Get keyboard for CURRENT language
        keyboard_layout = KEYBOARDS.get(self.current_language, KEYBOARDS["English"])
        
        for row in keyboard_layout:
            row_frame = ctk.CTkFrame(self.keyboard_frame, fg_color="transparent")
            row_frame.pack(fill="x", padx=5, pady=3)
            
            for char in row.split():
                if char == "␣":
                    btn = ctk.CTkButton(row_frame, text="SPACE", font=("Segoe UI", 9),
                                      width=120, height=35, fg_color="#34495E",
                                      hover_color=self.config.PRIMARY_COLOR,
                                      command=lambda: self._insert_char(" "))
                elif char == "⌫":
                    btn = ctk.CTkButton(row_frame, text="⌫", font=("Segoe UI", 12),
                                      width=60, height=35, fg_color=self.config.ACCENT_COLOR,
                                      hover_color="#C0392B", command=self._backspace_char)
                else:
                    btn = ctk.CTkButton(row_frame, text=char, font=("Segoe UI", 11),
                                      width=40, height=35, fg_color="#34495E",
                                      hover_color=self.config.PRIMARY_COLOR,
                                      command=lambda c=char: self._insert_char(c))
                btn.pack(side="left", padx=2)
    
    def _insert_char(self, char: str):
        current = self.query_entry.get()
        if len(current) < self.config.MAX_QUERY_LENGTH:
            self.query_entry.insert("end", char)
    
    def _backspace_char(self):
        current = self.query_entry.get()
        if current:
            self.query_entry.delete(len(current)-1, "end")
    
    def _update_char_count(self, event=None):
        count = len(self.query_entry.get())
        self.char_counter.configure(text=self._get_ui_text("char_count").format(current=count, max=500))
    
    def _upload_file(self):
        try:
            file_path = filedialog.askopenfilename(
                title="Select Crop Image",
                filetypes=[("Image Files", "*.png *.jpg *.jpeg")],
                initialdir=str(Path.home())
            )
            
            if file_path:
                validated_path = self.file_validator.validate_file(file_path)
                self.current_image_path = validated_path
                
                img = Image.open(validated_path)
                img.thumbnail((500, 500), Image.Resampling.LANCZOS)
                ctk_image = ctk.CTkImage(img, size=img.size)
                self.image_label.configure(image=ctk_image, text="")
                self.image_label.image = ctk_image
                
                self.status_label.configure(text=self._get_ui_text("success_upload"),
                                          text_color=self.config.PRIMARY_COLOR)
                self.logger.info(f"✅ Uploaded: {validated_path.name}")
        
        except FileValidationError as e:
            messagebox.showerror(self._get_ui_text("error_file"), str(e))
        except Exception as e:
            messagebox.showerror(self._get_ui_text("error_general"), str(e))
    
    def _toggle_camera(self):
        if self.camera_active:
            self._stop_camera()
        else:
            self._start_camera()
    
    def _start_camera(self):
        if self.camera_manager.start():
            self.camera_active = True
            self.camera_btn.configure(text=self._get_ui_text("capture_btn"),
                                     fg_color=self.config.PRIMARY_COLOR)
            self.status_label.configure(text=self._get_ui_text("status_camera_active"),
                                       text_color=self.config.SECONDARY_COLOR)
            self._stream_camera()
        else:
            messagebox.showerror(self._get_ui_text("error_general"), "Failed to open camera")
    
    def _stop_camera(self):
        self.camera_active = False
        self.camera_manager.stop()
        self.camera_btn.configure(text=self._get_ui_text("camera_btn"),
                                 fg_color=self.config.SECONDARY_COLOR)
        self.status_label.configure(text=self._get_ui_text("status_ready"),
                                   text_color=self.config.PRIMARY_COLOR)
    
    def _stream_camera(self):
        if not self.camera_active:
            return
        
        frame_data = self.camera_manager.get_frame()
        if frame_data:
            ret, frame = frame_data
            if ret:
                img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                img.thumbnail((500, 500), Image.Resampling.LANCZOS)
                ctk_image = ctk.CTkImage(img, size=img.size)
                self.image_label.configure(image=ctk_image, text="")
                self.image_label.image = ctk_image
                
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                self.current_image_path = self.config.TEMP_DIR / f"camera_{timestamp}.jpg"
                cv2.imwrite(str(self.current_image_path), frame)
        
        if self.camera_active:
            self.after(self.config.STREAM_UPDATE_INTERVAL_MS, self._stream_camera)
    
    def _diagnose(self):
        try:
            if self.is_analyzing:
                return
            
            if self.camera_active:
                self._stop_camera()
            
            if not self.current_image_path or not self.current_image_path.exists():
                messagebox.showwarning(self._get_ui_text("error_no_image"),
                                      self._get_ui_text("error_no_image_msg"))
                return
            
            query = self.input_validator.validate_query(self.query_entry.get())
            
            # Start analysis
            self.is_analyzing = True
            self.diagnose_btn.configure(state="disabled", fg_color="#95A5A6")
            self.status_label.configure(text=self._get_ui_text("status_analyzing"),
                                       text_color=self.config.SECONDARY_COLOR)
            self.output_textbox.delete("1.0", "end")
            self.output_textbox.insert("1.0", self._get_ui_text("analyzing_message"))
            
            # Run in thread
            thread = Thread(target=self._analysis_task, args=(self.current_image_path, query), daemon=True)
            thread.start()
        
        except InputValidationError as e:
            messagebox.showerror(self._get_ui_text("error_input"), str(e))
        except Exception as e:
            messagebox.showerror(self._get_ui_text("error_general"), str(e))
    
    def _analysis_task(self, image_path: Path, query: str):
        """Background analysis"""
        try:
            result = self.api_client.analyze_image(image_path, query, self.current_language)
            self.after(0, lambda: self._display_results(result))
        except APIConnectionError as e:
            self.after(0, lambda: self._display_error(str(e)))
        except Exception as e:
            self.after(0, lambda: self._display_error(f"Error: {str(e)}"))
        finally:
            self.after(0, self._reset_analysis_state)
    
    def _display_results(self, result: str):
        self.output_textbox.delete("1.0", "end")
        self.output_textbox.insert("1.0", result)
        self.status_label.configure(text="✅ " + ("Analysis complete!" if self.current_language == "English" else "विश्लेषण पूर्ण!"),
                                   text_color=self.config.PRIMARY_COLOR)
    
    def _display_error(self, error: str):
        self.output_textbox.delete("1.0", "end")
        self.output_textbox.insert("1.0", f"❌ ERROR:\n\n{error}")
        self.status_label.configure(text="❌ " + ("Failed" if self.current_language == "English" else "विफल"),
                                   text_color=self.config.ACCENT_COLOR)
    
    def _reset_analysis_state(self):
        self.is_analyzing = False
        self.diagnose_btn.configure(state="normal", fg_color=self.config.PRIMARY_COLOR)
    
    def _clear_screen(self):
        for widget in self.winfo_children():
            widget.destroy()
    
    def on_closing(self):
        try:
            self.logger.info("🔌 Closing...")
            self.camera_manager.stop()
            self.destroy()
        except:
            self.destroy()

# ============================================================================
# ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    try:
        print("=" * 70)
        print("🌾 AI CROP DISEASE DIAGNOSIS SYSTEM - MULTI-LANGUAGE")
        print("=" * 70)
        print("\n📋 CHECKLIST:")
        print("1. Ollama running: ollama serve")
        print("2. Model installed: ollama pull llava:7b")
        print("3. Camera permissions granted")
        print("\n✨ FEATURES:")
        print("• 11 Indian Languages with FULL UI translation")
        print("• Language-specific keyboards (Gujarati keyboard for Gujarati!)")
        print("• Accurate multi-language disease diagnosis")
        print("• Zoom slider for results")
        print("\n🚀 Launching...\n")
        
        app = CropDiagnosisApp()
        app.protocol("WM_DELETE_WINDOW", app.on_closing)
        app.mainloop()
    except Exception as e:
        logger.critical(f"🔥 Critical: {e}")
        raise